In [1]:
import pandas as pd

# 2013/14 (first season after Ferguson) to 2025/26
seasons = ["1314", "1415", "1516", "1617", "1718", "1819", "1920",
           "2021", "2122", "2223", "2324", "2425", "2526"]

frames = []
for s in seasons:
    url = f"https://www.football-data.co.uk/mmz4281/{s}/E0.csv"
    season = pd.read_csv(url, usecols=["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"], encoding="latin1")
    season["season"] = "20" + s[:2] + "/" + s[2:]
    frames.append(season)

df = pd.concat(frames, ignore_index=True)
df = df.dropna()  # some files have empty rows at the end

# save a copy so we don't download again
df.to_csv("data/epl_matches.csv", index=False)
df.shape

(4940, 7)

In [8]:
# home team's view of each match
home = pd.DataFrame({
    "season": df["season"],
    "team": df["HomeTeam"],
    "venue": "Home",
    "goals_for": df["FTHG"],
    "goals_against": df["FTAG"]
})

# away team's view of each match
away = pd.DataFrame({
    "season": df["season"],
    "team": df["AwayTeam"],
    "venue": "Away",
    "goals_for": df["FTAG"],
    "goals_against": df["FTHG"]
})

games = pd.concat([home, away], ignore_index=True)

# 3 for a win, 1 for a draw, 0 for a loss
games["points"] = 0
games.loc[games["goals_for"] > games["goals_against"], "points"] = 3
games.loc[games["goals_for"] == games["goals_against"], "points"] = 1

games[games["team"] == "Man United"].shape

(494, 6)

In [3]:
# add up every team's season
table = games.groupby(["season", "team"])[["points", "goals_for", "goals_against"]].sum().reset_index()
table["goal_diff"] = table["goals_for"] - table["goals_against"]

# sort like the real table: points, then goal difference, then goals scored
table = table.sort_values(["season", "points", "goal_diff", "goals_for"],
                          ascending=[True, False, False, False])
table["position"] = table.groupby("season").cumcount() + 1

united = table[table["team"] == "Man United"]
united

,season,team,points,goals_for,goals_against,goal_diff,position
10,2013/14,Man United,64,64.0,43.0,21.0,7
30,2014/15,Man United,70,62.0,37.0,25.0,4
49,2015/16,Man United,66,49.0,35.0,14.0,5
70,2016/17,Man United,69,54.0,29.0,25.0,6
91,2017/18,Man United,81,68.0,28.0,40.0,2
113,2018/19,Man United,66,65.0,54.0,11.0,6
131,2019/20,Man United,66,66.0,36.0,30.0,3
152,2020/21,Man United,74,73.0,44.0,29.0,2
172,2021/22,Man United,58,57.0,57.0,0.0,6
193,2022/23,Man United,75,58.0,43.0,15.0,3


In [4]:
# united's points and 4th place points, by season
united_pts = united.set_index("season")["points"]
fourth_pts = table[table["position"] == 4].set_index("season")["points"]

q1 = pd.DataFrame({"united": united_pts, "fourth_place": fourth_pts})
q1["gap"] = q1["united"] - q1["fourth_place"]
q1

,united,fourth_place,gap
season,,,
2013/14,64,79,-15
2014/15,70,70,0
2015/16,66,66,0
2016/17,69,76,-7
2017/18,81,75,6
2018/19,66,71,-5
2019/20,66,66,0
2020/21,74,67,7
2021/22,58,71,-13


### Q1: How have United done against the top 4 since Ferguson?

- United finished in the top 4 in 6 of 13 seasons
- 2024/25 was the worst, 27 points below 4th place
- 2015/16 was the closest miss, level on points with 4th but behind on goal difference
- United have been up and down every couple of seasons, with no steady improvement

In [5]:
utd_games = games[games["team"] == "Man United"]

# average points per game, home vs away, each season
q2 = utd_games.groupby(["season", "venue"])["points"].mean().unstack().round(2)
q2["difference"] = q2["Home"] - q2["Away"]
q2

venue,Away,Home,difference
season,,,
2013/14,1.79,1.58,-0.21
2014/15,1.37,2.32,0.95
2015/16,1.32,2.16,0.84
2016/17,1.84,1.79,-0.05
2017/18,1.79,2.47,0.68
2018/19,1.58,1.89,0.31
2019/20,1.53,1.95,0.42
2020/21,2.26,1.63,-0.63
2021/22,1.21,1.84,0.63


### Q2: How much better are United at home than away?

- United got more points per game at home in 10 of 13 seasons
- The biggest gap was 2022/23, with 2.53 points per game at home vs 1.42 away
- 2024/25 was the worst away season, under 1 point per game
- 2020/21 was the only big exception (better away than home), which was the season played in empty stadiums during COVID

In [6]:
q3 = united.set_index("season")[["goals_for", "goals_against", "goal_diff"]].astype(int)
q3

,goals_for,goals_against,goal_diff
season,,,
2013/14,64,43,21
2014/15,62,37,25
2015/16,49,35,14
2016/17,54,29,25
2017/18,68,28,40
2018/19,65,54,11
2019/20,66,36,30
2020/21,73,44,29
2021/22,57,57,0


### Q3: Are United scoring less or conceding more?

- In top 4 seasons, United scored 58 to 73 goals and conceded 28 to 50
- In the 3 worst seasons (2021/22, 2023/24, 2024/25), they scored 44 to 57 and conceded 54 to 58
- Both got worse, but conceding went up more than scoring went down
- 2023/24 and 2024/25 were the only seasons with a negative goal difference
- 2017/18 had the best defence, only 28 conceded

In [7]:
# one row per season with everything for the 3 charts
summary = pd.DataFrame({
    "points": q1["united"],
    "fourth_place": q1["fourth_place"],
    "home_ppg": q2["Home"],
    "away_ppg": q2["Away"],
    "goals_for": q3["goals_for"],
    "goals_against": q3["goals_against"]
})
summary.index.name = "season"
summary = summary.reset_index()

summary.to_csv("data/united_seasons.csv", index=False)
summary

,season,points,fourth_place,home_ppg,away_ppg,goals_for,goals_against
0,2013/14,64,79,1.58,1.79,64,43
1,2014/15,70,70,2.32,1.37,62,37
2,2015/16,66,66,2.16,1.32,49,35
3,2016/17,69,76,1.79,1.84,54,29
4,2017/18,81,75,2.47,1.79,68,28
5,2018/19,66,71,1.89,1.58,65,54
6,2019/20,66,66,1.95,1.53,66,36
7,2020/21,74,67,1.63,2.26,73,44
8,2021/22,58,71,1.84,1.21,57,57
9,2022/23,75,71,2.53,1.42,58,43
